# nQPU-Metal JAX Integration Demo

This notebook demonstrates the JAX integration for nQPU-Metal quantum simulator.

**Features:**
- Automatic differentiation via parameter-shift rule
- Efficient batch execution with VMAP
- Quantum kernels for machine learning
- Variational Quantum Eigensolver (VQE)
- Quantum natural gradient optimization

In [ ]:
# Setup
import sys
import numpy as np
import matplotlib.pyplot as plt

# Add nqpu-metal python module to path
sys.path.insert(0, '/path/to/nqpu-metal/python')  # Update this path

import jax
import jax.numpy as jnp
from jax import grad, jit, vmap

print(f"JAX version: {jax.__version__}")
print(f"JAX backend: {jax.default_backend()}")

In [ ]:
# Import nQPU-Metal JAX integration
from nqpu_jax import (
    quantum_expectation,
    vmap_quantum,
    quantum_kernel_matrix,
    make_vqe_loss,
    check_installation,
)

# Check installation
status = check_installation()
print(f"Installation status: {status}")
assert all(status.values()), "Missing dependencies!"

## 1. Basic Circuit with Automatic Differentiation

In [ ]:
# Define a simple 2-qubit circuit
circuit = {
    'n_qubits': 2,
    'gates': [
        ('H', 0),              # Hadamard on qubit 0
        ('RY', 0, 'theta'),    # Parameterized Y rotation
        ('RY', 1, 'phi'),      # Second parameter
        ('CNOT', 0, 1),        # Entangling gate
    ],
    'observable': 'Z0'         # Measure ⟨Z₀⟩
}

# Parameters
params = jnp.array([0.5, 1.2])

# Compute expectation value
exp_val = quantum_expectation(params, circuit)
print(f"Expectation value ⟨Z₀⟩ = {exp_val:.6f}")

# Compute gradients automatically
grad_fn = grad(quantum_expectation, argnums=0)
grads = grad_fn(params, circuit)
print(f"Gradients ∇⟨Z₀⟩ = {grads}")

### Visualize Expectation Landscape

In [ ]:
# Create parameter grid
theta_range = np.linspace(0, 2*np.pi, 50)
phi_range = np.linspace(0, 2*np.pi, 50)
Theta, Phi = np.meshgrid(theta_range, phi_range)

# Compute expectations over grid
expectations = np.zeros_like(Theta)
for i in range(len(theta_range)):
    for j in range(len(phi_range)):
        params_ij = jnp.array([Theta[i, j], Phi[i, j]])
        expectations[i, j] = quantum_expectation(params_ij, circuit)

# Plot
plt.figure(figsize=(10, 8))
plt.contourf(Theta, Phi, expectations, levels=20, cmap='RdBu_r')
plt.colorbar(label='⟨Z₀⟩')
plt.xlabel('θ')
plt.ylabel('φ')
plt.title('Expectation Value Landscape')
plt.scatter([0.5], [1.2], color='red', s=100, marker='*', label='Current params')
plt.legend()
plt.show()

## 2. Batch Execution with VMAP

In [ ]:
# Create batched function
batch_fn = vmap_quantum(circuit, observable='Z0')

# Batch of parameters
batch_params = jnp.array([
    [0.0, 0.0],
    [np.pi/4, np.pi/4],
    [np.pi/2, np.pi/2],
    [np.pi, np.pi],
])

# Execute batch (fast!)
results = batch_fn(batch_params)
print(f"Batch results: {results}")

# Visualize
plt.figure(figsize=(8, 5))
plt.plot(results, 'o-', markersize=10)
plt.xlabel('Batch index')
plt.ylabel('⟨Z₀⟩')
plt.title('Batch Quantum Expectations')
plt.grid(True, alpha=0.3)
plt.show()

## 3. Quantum Kernel for QSVM

In [ ]:
# Feature map circuit (4 qubits)
feature_map = {
    'n_qubits': 4,
    'gates': [
        ('H', 0), ('H', 1), ('H', 2), ('H', 3),
        ('RY', 0, 'x0'), ('RY', 1, 'x1'),
        ('RY', 2, 'x2'), ('RY', 3, 'x3'),
        ('CNOT', 0, 1), ('CNOT', 1, 2), ('CNOT', 2, 3),
    ]
}

# Generate sample data (2 classes)
np.random.seed(42)
X_class0 = np.random.randn(5, 4) * 0.5  # Cluster 1
X_class1 = np.random.randn(5, 4) * 0.5 + 2.0  # Cluster 2
X = jnp.vstack([X_class0, X_class1])

# Compute kernel matrix K[i,j] = |⟨ψ(xᵢ)|ψ(xⱼ)⟩|²
K = quantum_kernel_matrix(X, X, feature_map)

# Visualize kernel
plt.figure(figsize=(8, 6))
plt.imshow(K, cmap='viridis')
plt.colorbar(label='Kernel value')
plt.title('Quantum Kernel Matrix')
plt.xlabel('Sample j')
plt.ylabel('Sample i')
plt.axhline(4.5, color='red', linestyle='--', linewidth=2)
plt.axvline(4.5, color='red', linestyle='--', linewidth=2)
plt.text(2, 2, 'Class 0', ha='center', va='center', color='white', fontsize=12, weight='bold')
plt.text(7, 7, 'Class 1', ha='center', va='center', color='white', fontsize=12, weight='bold')
plt.show()

print(f"Kernel matrix shape: {K.shape}")
print(f"Within-class similarity: {np.mean(K[:5, :5]):.4f}")
print(f"Between-class similarity: {np.mean(K[:5, 5:]):.4f}")

## 4. Variational Quantum Eigensolver (VQE)

In [ ]:
# Ansatz circuit (2 qubits, 4 parameters)
ansatz = {
    'n_qubits': 2,
    'gates': [
        ('RY', 0, 'theta_0'),
        ('RY', 1, 'theta_1'),
        ('CNOT', 0, 1),
        ('RY', 0, 'theta_2'),
        ('RY', 1, 'theta_3'),
    ]
}

# Hamiltonian: H = 0.5·Z₀ + 0.5·Z₁
# Ground state should have energy ≈ -1.0 (both qubits in |1⟩)
hamiltonian = [
    ('Z0', 0.5),
    ('Z1', 0.5),
]

# Create VQE loss function
loss_fn = make_vqe_loss(ansatz, hamiltonian)

# Initial parameters (random)
params = jnp.array([0.1, 0.2, 0.3, 0.4])

# Gradient descent optimization
learning_rate = 0.1
n_steps = 100
energy_history = []

for step in range(n_steps):
    energy = loss_fn(params)
    energy_history.append(float(energy))
    
    grads = grad(loss_fn)(params)
    params = params - learning_rate * grads

# Plot convergence
plt.figure(figsize=(10, 5))
plt.plot(energy_history, 'b-', linewidth=2)
plt.axhline(-1.0, color='red', linestyle='--', label='True ground state')
plt.xlabel('Iteration')
plt.ylabel('Energy')
plt.title('VQE Optimization Convergence')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f"Final energy: {energy_history[-1]:.6f}")
print(f"Final parameters: {params}")

## 5. JIT Compilation Performance

In [ ]:
import time

# Test circuit
test_circuit = {
    'n_qubits': 4,
    'gates': [
        ('RY', 0, 'p0'), ('RY', 1, 'p1'),
        ('RY', 2, 'p2'), ('RY', 3, 'p3'),
        ('CNOT', 0, 1), ('CNOT', 1, 2), ('CNOT', 2, 3),
    ],
    'observable': 'Z0'
}

params = jnp.array([0.1, 0.2, 0.3, 0.4])

# Regular function
def regular_fn(p):
    return quantum_expectation(p, test_circuit)

# JIT-compiled function
@jit
def jit_fn(p):
    return quantum_expectation(p, test_circuit)

# Benchmark
n_runs = 1000

# Regular
start = time.time()
for _ in range(n_runs):
    _ = regular_fn(params)
time_regular = time.time() - start

# JIT (warm up first)
_ = jit_fn(params)  # Compilation
start = time.time()
for _ in range(n_runs):
    _ = jit_fn(params)
time_jit = time.time() - start

# Results
print(f"Regular: {n_runs} runs in {time_regular:.3f}s ({n_runs/time_regular:.0f} runs/sec)")
print(f"JIT:     {n_runs} runs in {time_jit:.3f}s ({n_runs/time_jit:.0f} runs/sec)")
print(f"Speedup: {time_regular/time_jit:.2f}x")

# Visualize
plt.figure(figsize=(8, 5))
plt.bar(['Regular', 'JIT'], [time_regular, time_jit], color=['blue', 'green'])
plt.ylabel('Time (seconds)')
plt.title(f'Performance Comparison ({n_runs} runs)')
plt.show()

## 6. Gradient Descent Trajectory Visualization

In [ ]:
# Simple 2-parameter circuit for visualization
simple_circuit = {
    'n_qubits': 1,
    'gates': [
        ('RY', 0, 'theta'),
        ('RZ', 0, 'phi'),
    ],
    'observable': 'Z0'
}

# Loss function: minimize ⟨Z₀⟩
def loss(p):
    return quantum_expectation(p, simple_circuit)

# Create landscape
theta_range = np.linspace(0, 2*np.pi, 100)
phi_range = np.linspace(0, 2*np.pi, 100)
Theta, Phi = np.meshgrid(theta_range, phi_range)

Z = np.zeros_like(Theta)
for i in range(100):
    for j in range(100):
        Z[i, j] = loss(jnp.array([Theta[i, j], Phi[i, j]]))

# Gradient descent trajectory
params = jnp.array([0.5, 0.5])
trajectory = [params]

for _ in range(50):
    grads = grad(loss)(params)
    params = params - 0.1 * grads
    trajectory.append(params)

trajectory = jnp.array(trajectory)

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.contourf(Theta, Phi, Z, levels=30, cmap='viridis')
plt.colorbar(label='⟨Z₀⟩')
plt.plot(trajectory[:, 0], trajectory[:, 1], 'r.-', linewidth=2, markersize=8)
plt.scatter(trajectory[0, 0], trajectory[0, 1], color='green', s=200, marker='*', label='Start', zorder=10)
plt.scatter(trajectory[-1, 0], trajectory[-1, 1], color='red', s=200, marker='*', label='End', zorder=10)
plt.xlabel('θ')
plt.ylabel('φ')
plt.title('Gradient Descent Trajectory')
plt.legend()

plt.subplot(1, 2, 2)
losses = [float(loss(p)) for p in trajectory]
plt.plot(losses, 'b-', linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Loss Convergence')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final loss: {losses[-1]:.6f}")
print(f"Final params: {trajectory[-1]}")

## Summary

This notebook demonstrated:

✅ **Automatic differentiation** via JAX custom VJP and parameter-shift rule  
✅ **Batch execution** with VMAP for efficient parallel processing  
✅ **Quantum kernels** for quantum machine learning (QSVM)  
✅ **VQE optimization** for finding ground states  
✅ **JIT compilation** for performance optimization  
✅ **Gradient descent** visualization on quantum landscapes  

The JAX integration provides a **seamless interface** between Python/JAX and high-performance Rust quantum simulation.

### Next Steps

- Try different ansatz circuits
- Implement quantum natural gradient
- Build quantum neural networks
- Explore quantum chemistry applications
- Benchmark on larger systems (>10 qubits)

See `README_JAX.md` for full API documentation.